In [1]:
import xarray as xr
from pystac_client import Client

catalog = Client.open("https://eoapi.workspace.earthcode.eox.at/stac/")
item = next(catalog.search(collections=["sentinel1-amazonas-forest-loss"]).items())
display(item)

ds = xr.open_zarr(item.assets["data"].href)
ds

<Item id=s14scienceamazonas>

<xarray.Dataset> Size: 22TB
Dimensions:            (time: 194, lat: 146501, lon: 190847)
Coordinates:
  * time               (time) datetime64[ns] 2kB 2017-01-02 ... 2024-11-29
  * lat                (lat) float64 1MB 9.047 9.046 9.046 ... -17.27 -17.27
  * lon                (lon) float64 2MB -79.2 -79.2 -79.2 ... -44.91 -44.91
    spatial_ref        int64 8B ...
Data variables:
    tree_cover_change  (time, lat, lon) float32 22TB dask.array<chunksize=(1, 5120, 5120), meta=np.ndarray>
Attributes:
    geospatial_lon_resolution:  0.00017966
    geospatial_lat_resolution:  0.00017966
    description:                The Sentinel-1 for Science Amazonas project d...
    license:                    Creative Commons Attribution 4.0 Internationa...
    license_url:                https://creativecommons.org/licenses/by/4.0/
    references:                 https://sen4ama.gisat.cz/ https://eo4society....
    platform:                   Sentinel-1
    project:                    Sentinel-1 for Science Amazonas
    keywords:                   Forests, Deforestation, SAR, Amazon, Sentinel-1

In [2]:
# note lat goes from max to min
min_x, min_y, max_x, max_y = (-64.8, -7.3, -64.7, -7.2)

In [3]:
aoi = ds.sel(lon=slice(min_x, max_x), lat=slice(max_y, min_y), time=slice('2017-01-02', '2021-01-14'))
aoi

<xarray.Dataset> Size: 153MB
Dimensions:            (time: 123, lat: 557, lon: 557)
Coordinates:
  * time               (time) datetime64[ns] 984B 2017-01-02 ... 2021-01-05
  * lat                (lat) float64 4kB -7.2 -7.2 -7.2 ... -7.3 -7.3 -7.3
  * lon                (lon) float64 4kB -64.8 -64.8 -64.8 ... -64.7 -64.7 -64.7
    spatial_ref        int64 8B ...
Data variables:
    tree_cover_change  (time, lat, lon) float32 153MB dask.array<chunksize=(1, 557, 557), meta=np.ndarray>
Attributes:
    geospatial_lon_resolution:  0.00017966
    geospatial_lat_resolution:  0.00017966
    description:                The Sentinel-1 for Science Amazonas project d...
    license:                    Creative Commons Attribution 4.0 Internationa...
    license_url:                https://creativecommons.org/licenses/by/4.0/
    references:                 https://sen4ama.gisat.cz/ https://eo4society....
    platform:                   Sentinel-1
    project:                    Sentinel-1 for Science Amazonas
    keywords:                   Forests, Deforestation, SAR, Amazon, Sentinel-1

In [4]:
aoi = aoi.compute()

In [5]:
import numpy as np
ds_sum = aoi['tree_cover_change'].sum(dim='time', skipna=True)

In [6]:
import hvplot.xarray
import cartopy.crs as ccrs
ds_sum.where(ds_sum > 0, drop=True).hvplot(
    x='lon', 
    y='lat', 
    crs=ccrs.PlateCarree(),
    tiles='OSM',
    title=f'Total change {aoi.time.min().values} - {aoi.time.max().values}',
    width=600,
    height=500
)

:Overlay
   .WMTS.I  :WMTS   [Longitude,Latitude]
   .Image.I :Image   [lon,lat]   (Forest tree cover change detection)